In [1]:
import os
import json
import random
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass, field

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import cvxpy as cp

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU Enabled: {len(gpus)} device(s)")
    except Exception as e:
        print(f"⚠️  GPU configuration warning: {e}, continuing...")
else:
    print("⚠️  No GPU detected, using CPU")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

def get_profile_constraints(profile: str) -> Dict[str, Any]:
        profile = profile.lower()
        if profile == 'conservative':
            return {
                'expected_return_multiplier': 0.40,  # dampen returns strongly
                'multiplier_power': 0.85,            # compress differences
                'risk_aversion': 7.0,                # very risk averse
                'max_weight': 0.10,                  # small single-asset cap
                'shrinkage': 0.25,                   # even stronger shrink
                'entropy_penalty': 5e-3              # stronger smoothing to diversify
            }
        elif profile == 'aggressive':
            return {
                'expected_return_multiplier': 2.5,   # push expected returns harder
                'multiplier_power': 1.8,            # non-linear amplification
                'risk_aversion': 0.8,               # still much less risk averse
                'max_weight': 0.18,                 #  cap any single asset at 20%, no 45% bonds
                'shrinkage': 0.05,                  # modest shrink
                'entropy_penalty': 1e-3             # small but non-zero smoothing
            }
        else:
            raise ValueError("Unknown profile: choose 'conservative' or 'aggressive'.")


@dataclass
class ResearchConfig:
    """Production-Ready Multi-Asset Portfolio Configuration"""
    
    g7_indices: Dict[str, str] = field(default_factory=lambda: {
        # ===== US LARGE CAP (5) =====
        'AAPL': 'AAPL',
        'MSFT': 'MSFT',
        'GOOGL': 'GOOGL',
        'AMZN': 'AMZN',
        'META': 'META',
        
        # ===== US MID/GROWTH (5) =====
        'NVDA': 'NVDA',
        'TSLA': 'TSLA',
        'V': 'V',
        'JPM': 'JPM',
        'WMT': 'WMT',
        
        # ===== INDIA LARGE CAP (5) =====
        'RELIANCE': 'RELIANCE.NS',
        'TCS': 'TCS.NS',
        'HDFCBANK': 'HDFCBANK.NS',
        'INFY': 'INFY.NS',
        'ITC': 'ITC.NS',
        
        # ===== EQUITY ETFs (5) =====
        'SPY': 'SPY',
        'QQQ': 'QQQ',
        'IWM': 'IWM',
        'EFA': 'EFA',
        'VTI': 'VTI',
        
        # ===== BONDS (5) =====
        'AGG': 'AGG',
        'TLT': 'TLT',
        'IEF': 'IEF',
        'LQD': 'LQD',
        'HYG': 'HYG',
        
        # ===== REITs (3) =====
        'VNQ': 'VNQ',
        'O': 'O',
        'PLD': 'PLD',
        
        # ===== COMMODITIES (4) =====
        'GLD': 'GLD',
        'SLV': 'SLV',
        'USO': 'USO',
        'DBA': 'DBA',
        
        # ===== SECTORS (4) =====
        'XLK': 'XLK',
        'XLF': 'XLF',
        'XLV': 'XLV',
        'XLE': 'XLE',
        
        # ===== DIVIDEND/VALUE (4) =====
        'VYM': 'VYM',
        'SCHD': 'SCHD',
        'DVY': 'DVY',
        'VTV': 'VTV',
    })
    
    years_of_data: int = 10
    train_window_days: int = 504
    validation_window: int = 63
    
    pred_horizons: List[int] = field(default_factory=lambda: [5])
    
    # Trading parameters
    rebalance_freq: int = 10
    retrain_cadence: int = 40
    commission_rate: float = 0.0005
    transaction_slippage: float = 0.0002  # NEW: Realistic slippage
    min_trade_size: float = 0.005  # NEW: Don't trade if change < 1%
    rebalance_threshold: float = 0.02  # NEW: Only rebalance if drift > 2%
    
    # Risk management
    ewma_span: int = 40
    risk_aversion: float = 2.0
    risk_free_annual: float = 0.05
    max_position: float = 0.15
    min_position: float = 0.0
    risk_profile: str = "conservative"
    
    # LSTM parameters
    lstm_epochs: int = 35
    lstm_batch: int = 64
    lstm_lookback: int = 180
    lstm_patience: int = 12
    lstm_architecture: List[int] = field(default_factory=lambda: [128, 64, 32])
    
    # Feature engineering
    ma_periods: List[int] = field(default_factory=lambda: [5, 10, 20, 60])
    vol_periods: List[int] = field(default_factory=lambda: [10, 20, 60])
    mom_periods: List[int] = field(default_factory=lambda: [5, 10, 20])
    
    # Portfolio settings
    enable_ensemble: bool = True
    ensemble_lstm_weight: float = 0.75
    enable_volatility_targeting: bool = True
    target_volatility: float = 0.10
    bootstrap_iterations: int = 1000
    
    # Crisis detection
    crisis_vol_threshold: float = 0.05
    
    # Out-of-sample testing
    out_of_sample_periods: List[Dict[str, str]] = field(default_factory=lambda: [
        {
            'name': 'Bull_Market_2023_2025',
            'train_start': '2019-01-01',
            'train_end': '2022-12-31',
            'test_start': '2023-01-01',
            'test_end': '2025-09-30'
        },
        {
            'name': 'COVID_Recovery_2020_2022',
            'train_start': '2016-01-01',
            'train_end': '2019-12-31',
            'test_start': '2020-01-01',
            'test_end': '2022-12-31'
        }
    ])
    
    output_dir: str = "publication_output_production"
    use_synthetic: bool = False
    verbose: bool = True
    
    @property
    def tickers(self):
        return list(self.g7_indices.keys())
    
    @property
    def ticker_symbols(self):
        return list(self.g7_indices.values())
    
    @property
    def risk_free_daily(self):
        return self.risk_free_annual / 252.0

    @property
    def live_weights_dir(self) -> str:
        return os.path.join(self.output_dir, "live_weights")

    def configure_for_profile(self, profile: str):
        """Apply high-level risk settings based on a simple profile name."""
        p = profile.strip().lower()
        if p in ("c", "conservative", "normal"):
            self.risk_profile = "conservative"
            # more risk-averse: lower vol target, tighter caps, higher gamma
            self.risk_aversion = 3.0
            self.target_volatility = 0.10
            self.max_position = 0.15
        elif p in ("a", "aggressive", "high"):
            self.risk_profile = "aggressive"
            # more return-seeking: higher vol target, looser caps, lower gamma
            self.risk_aversion = 1.0
            self.target_volatility = 0.20
            self.max_position = 0.30
        else:
            print(f"Unknown risk profile '{profile}', defaulting to conservative.")
            self.risk_profile = "conservative"
            self.risk_aversion = 3.0
            self.target_volatility = 0.10
            self.max_position = 0.15
    
    def create_directories(self):
        for subdir in ['', 'plots', 'models', 'data', 'out_of_sample', 'statistical_tests', 'live_weights']:
            os.makedirs(os.path.join(self.output_dir, subdir), exist_ok=True)


class DataManager:
    """Data management with health checks"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        
    def check_asset_health(self, ticker: str, price_series: pd.Series) -> bool:
        """Minimal health check"""
        if len(price_series) < 50:
            return False
        return True

        
    def download_data(self):
        """Download and validate multi-asset data"""
        end = datetime.today()
        start = end - timedelta(days=365 * self.config.years_of_data + 60)
        
        print(f"\n{'='*70}")
        print(f"DOWNLOADING & VALIDATING MULTI-ASSET DATA")
        print(f"{'='*70}")
        print(f"Period: {start.date()} to {end.date()}")
        print(f"Assets: {len(self.config.ticker_symbols)}\n")
        
        cache = os.path.join(self.config.output_dir, "data", "multiasset_prices.csv")
        if os.path.exists(cache):
            try:
                df = pd.read_csv(cache, index_col=0, parse_dates=True)
                if len(df) >= 500:
                    print(f"✓ Loaded {len(df)} days from cache")
                    print(f"✓ {len(df.columns)} assets available\n")
                    return df
            except:
                pass
        
        try:
            print("Downloading from Yahoo Finance...")
            df_all = yf.download(
                tickers=self.config.ticker_symbols,
                start=start, end=end,
                progress=False, auto_adjust=False, group_by='ticker'
            )
            
            data_dict = {}
            healthy_count = 0
            
            for ticker_name, ticker_symbol in self.config.g7_indices.items():
                try:
                    if len(self.config.ticker_symbols) > 1:
                        adj_close = df_all[ticker_symbol]['Adj Close']
                    else:
                        adj_close = df_all['Adj Close']
                    
                    # NEW: Health check
                    if self.check_asset_health(ticker_name, adj_close):
                        data_dict[ticker_name] = adj_close
                        healthy_count += 1
                        print(f"✓ {ticker_name:20s} - Healthy")
                except Exception as e:
                    print(f"✗ {ticker_name:20s}: {e}")
            
            df = pd.DataFrame(data_dict).dropna()
            df.to_csv(cache)
            print(f"\n✓ Downloaded {len(df)} days, {healthy_count}/{len(self.config.ticker_symbols)} healthy assets\n")
            return df
        except Exception as e:
            print(f"Download failed: {e}")
            raise
    
    def compute_features(self, price_df):
        """Enhanced feature engineering"""
        print("Computing features...")
        feat_list = []
        
        for ticker in price_df.columns:
            p = price_df[ticker]
            ret = p.pct_change().fillna(0)
            
            features = {
                (ticker, 'price'): p,
                (ticker, 'ret'): ret,
                (ticker, 'log_ret'): np.log(p / p.shift(1)).fillna(0),
            }
            
            for period in self.config.ma_periods:
                features[(ticker, f'ma{period}')] = p.rolling(period).mean()
                features[(ticker, f'ma{period}_ratio')] = p / p.rolling(period).mean() - 1
            
            for period in self.config.vol_periods:
                features[(ticker, f'vol{period}')] = ret.rolling(period).std()
            
            for period in self.config.mom_periods:
                features[(ticker, f'mom{period}')] = p / p.shift(period) - 1
            
            features[(ticker, 'rsi14')] = self._compute_rsi(ret, 14)
            features[(ticker, 'autocorr')] = ret.rolling(20).apply(lambda x: x.autocorr())
            
            features[(ticker, 'ema12')] = p.ewm(span=12).mean()
            features[(ticker, 'ema26')] = p.ewm(span=26).mean()
            features[(ticker, 'macd')] = features[(ticker, 'ema12')] - features[(ticker, 'ema26')]
            
            bb_ma = p.rolling(20).mean()
            bb_std = p.rolling(20).std()
            features[(ticker, 'bb_upper')] = (bb_ma + 2*bb_std) / p - 1
            features[(ticker, 'bb_lower')] = (bb_ma - 2*bb_std) / p - 1
            
            features[(ticker, 'regime')] = (p.rolling(60).mean() > p.rolling(120).mean()).astype(float)
            
            df = pd.DataFrame(features)
            feat_list.append(df)
        
        merged = pd.concat(feat_list, axis=1).sort_index()
        merged = merged.ffill().bfill().dropna()
        
        print(f"✓ {len(merged)} samples, {len(merged.columns)} features\n")
        return merged
    
    def _compute_rsi(self, returns, period=14):
        gains = returns.where(returns > 0, 0).rolling(period).mean()
        losses = -returns.where(returns < 0, 0).rolling(period).mean()
        rs = gains / (losses + 1e-10)
        return (100 - (100 / (1 + rs))).fillna(50)
    
    def create_labels(self, price_df):
        labels = {}
        for h in self.config.pred_horizons:
            labels[h] = price_df.shift(-h) / price_df - 1
        return labels


class LSTMPredictor:
    """LSTM predictor"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        self.models = {}
        self.scalers = {}
        
    def build_model(self, n_features):
        model = models.Sequential([
            layers.Input(shape=(self.config.lstm_lookback, n_features)),
            layers.LSTM(64, return_sequences=True),  # Reduced from 128
            layers.Dropout(0.3),                     # Increased from 0.2
            layers.LSTM(32),                         # Removed middle LSTM layer
            layers.Dropout(0.3),                     # Increased from 0.2
            layers.Dense(16, activation='relu'),     # Reduced from 32
            layers.Dense(1)                          # Removed extra Dense layer
        ])
        
        # Adaptive learning rate based on risk profile
        if self.config.risk_profile == "aggressive":
            learning_rate = 0.002  # Higher LR for more responsive predictions
        else:
            learning_rate = 0.001  # Standard LR for conservative
        
        # Adaptive learning rate based on risk profile
        if self.config.risk_profile == "aggressive":
            learning_rate = 0.002
        else:
            learning_rate = 0.001
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse',
            metrics=['mae']
        )
        return model
    
    def prepare_sequences(self, X, y):
        X_seq, y_seq = [], []
        for i in range(self.config.lstm_lookback, len(X)):
            X_seq.append(X[i-self.config.lstm_lookback:i])
            y_seq.append(y[i])
        return np.array(X_seq), np.array(y_seq)
    
    def train_model(self, ticker, horizon, X_train, y_train, X_val=None, y_val=None):
        try:
            if ticker not in self.scalers:
                self.scalers[ticker] = StandardScaler()
                X_train_scaled = self.scalers[ticker].fit_transform(X_train)
            else:
                X_train_scaled = self.scalers[ticker].transform(X_train)
            
            X_seq, y_seq = self.prepare_sequences(X_train_scaled, y_train)
            
            if len(X_seq) < 50:
                return None
            
            model = self.build_model(X_train.shape[1])
            
            early_stop = callbacks.EarlyStopping(
                monitor='val_loss' if X_val is not None else 'loss',
                patience=self.config.lstm_patience,
                restore_best_weights=True, verbose=0
            )
            
            val_data = None
            if X_val is not None and len(X_val) > self.config.lstm_lookback:
                X_val_scaled = self.scalers[ticker].transform(X_val)
                X_val_seq, y_val_seq = self.prepare_sequences(X_val_scaled, y_val)
                if len(X_val_seq) > 0:
                    val_data = (X_val_seq, y_val_seq)
            
            model.fit(X_seq, y_seq, epochs=self.config.lstm_epochs,
                     batch_size=self.config.lstm_batch,
                     validation_data=val_data, callbacks=[early_stop], verbose=0)
            
            if ticker not in self.models:
                self.models[ticker] = {}
            self.models[ticker][horizon] = model
            
            if self.config.verbose:
                print(f"    ✓ {ticker}")
            
            return model
        except Exception as e:
            if self.config.verbose:
                print(f"    ✗ {ticker}: {e}")
            return None
    
    def predict(self, ticker, horizon, X_recent):
        try:
            if ticker not in self.models or horizon not in self.models[ticker]:
                return 0.0
            X_scaled = self.scalers[ticker].transform(X_recent)
            if len(X_scaled) < self.config.lstm_lookback:
                return 0.0
            X_seq = X_scaled[-self.config.lstm_lookback:].reshape(1, self.config.lstm_lookback, -1)
            pred = self.models[ticker][horizon].predict(X_seq, verbose=0)
            return float(pred[0, 0])
        except:
            return 0.0

    


class PortfolioOptimizer:
    """Portfolio optimization with app features"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        self.current_weights = None
        self.predictions = {}
        self.current_volatility = None
        self.current_drawdown = 0.0
        self.crisis_mode = False
        
    def should_rebalance(self, current_weights: Dict, target_weights: Dict) -> bool:
        """NEW: Check if rebalancing is needed"""
        if current_weights is None:
            return True
        
        max_drift = 0.0
        for ticker in target_weights.keys():
            drift = abs(target_weights[ticker] - current_weights.get(ticker, 0))
            max_drift = max(max_drift, drift)
        
        return max_drift > self.config.rebalance_threshold

    def should_rebalance_signal(self, prev_mu: Dict[str, float], new_mu: Dict[str, float], threshold: float = None) -> bool:
        """
        Rebalance if the expected-return signal meaningfully changed.
        Uses mean absolute change in mu across tickers.
        Auto-calibrates threshold based on signal scale.
        """
        if prev_mu is None:
            return True
        
        # Only compare tickers that exist in both dictionaries
        common_keys = set(new_mu.keys()) & set(prev_mu.keys())
        
        if not common_keys:
            return True  # No overlap, must rebalance
        
        diffs = []
        for k in common_keys:
            diffs.append(abs(new_mu[k] - prev_mu[k]))
        
        # Auto-calibrate threshold if not provided
        if threshold is None:
            all_values = list(new_mu.values()) + list(prev_mu.values())
            threshold = np.std(all_values) * 0.6  # 0.5 standard deviations
            threshold = np.clip(threshold, 0.04, 0.12)  # Bound between 4-12%
        
        return (np.mean(diffs) > threshold)
    
    def round_portfolio_to_dollars(self, weights: Dict, total_value: float) -> Dict:
        """NEW: Convert weights to dollar amounts"""
        dollar_amounts = {}
        for ticker, weight in weights.items():
            amount = weight * total_value
            # Round to nearest $10 for practicality
            rounded = round(amount / 10) * 10
            dollar_amounts[ticker] = rounded
        
        # Ensure total equals portfolio value
        total_allocated = sum(dollar_amounts.values())
        if total_allocated > 0:
            scale = total_value / total_allocated
            dollar_amounts = {t: amt * scale for t, amt in dollar_amounts.items()}
        
        return dollar_amounts
    
    def explain_allocation(self, weights: Dict) -> Dict:
        """NEW: Explain portfolio decisions"""
        explanations = {}
        
        for ticker, weight in weights.items():
            pred = self.predictions.get(ticker, 0.0)
            
            if weight > 0.12:
                reason = f"High conviction: LSTM predicts +{pred*100:.1f}% return"
            elif weight > 0.08:
                reason = f"Moderate position: Positive signal (+{pred*100:.1f}%)"
            elif weight > 0.03:
                reason = f"Small position: Diversification holding"
            elif weight > 0:
                reason = f"Minimal weight: Weak signal or defensive"
            else:
                reason = "Zero weight: Negative outlook or excluded"
            
            explanations[ticker] = {
                'weight': weight,
                'prediction': pred,
                'reason': reason
            }
        
        return explanations
    
    def generate_risk_alerts(self) -> List[str]:
        """NEW: Generate portfolio risk warnings"""
        alerts = []
        
        if self.current_weights is None:
            return alerts
        
        # Concentration risk
        max_weight = max(self.current_weights.values())
        if max_weight > 0.20:
            alerts.append(f"⚠️ Concentration Risk: {max_weight*100:.1f}% in single asset")
        
        # Volatility alert
        if self.current_volatility and self.current_volatility > 0.20:
            alerts.append(f"⚠️ High Volatility: Portfolio vol at {self.current_volatility*100:.1f}%")
        
        # Drawdown warning
        if self.current_drawdown < -0.10:
            alerts.append(f"⚠️ Drawdown Alert: Portfolio down {abs(self.current_drawdown)*100:.1f}% from peak")
        
        # Crisis mode
        if self.crisis_mode:
            alerts.append("🛡️ Crisis Mode: Defensive positioning active")
        
        return alerts
    
    def get_current_portfolio_weights(self) -> Dict:
        """NEW: Get live trading weights (for app API)"""
        return {
            'weights': self.current_weights if self.current_weights else {},
            'last_updated': datetime.now().isoformat(),
            'predictions': self.predictions,
            'confidence': 'high' if not self.crisis_mode else 'low',
            'risk_level': 'defensive' if self.crisis_mode else 'normal',
            'volatility': float(self.current_volatility) if self.current_volatility else None,
            'alerts': self.generate_risk_alerts()
        }
    
    def analyze_attribution(self, returns: pd.DataFrame, weights_history: List) -> Dict:
        """NEW: Performance attribution analysis"""
        if len(weights_history) == 0:
            return {}
        
        attribution = {}
        for ticker in returns.columns:
            ticker_returns = returns[ticker].values
            ticker_weights = np.array([w.get(ticker, 0) for w in weights_history])
            
            # Ensure same length
            min_len = min(len(ticker_returns), len(ticker_weights))
            contribution = np.sum(ticker_returns[:min_len] * ticker_weights[:min_len])
            
            attribution[ticker] = {
                'contribution': contribution,
                'avg_weight': ticker_weights[:min_len].mean(),
                'return': ticker_returns[:min_len].mean()
            }
        
        # Sort by contribution
        sorted_attr = sorted(attribution.items(), key=lambda x: x[1]['contribution'], reverse=True)
        return dict(sorted_attr)
    
    def ewma_covariance(self, returns_df):
        alpha = 2.0 / (self.config.ewma_span + 1.0)
        weights = np.array([(1-alpha)**(len(returns_df)-1-i) for i in range(len(returns_df))])
        weights /= weights.sum()
        
        demeaned = returns_df - returns_df.mean()
        cov = np.zeros((returns_df.shape[1], returns_df.shape[1]))
        
        for i, w in enumerate(weights):
            r = demeaned.iloc[i].values.reshape(-1, 1)
            cov += w * (r @ r.T)
        
        min_eig = np.min(np.linalg.eigvals(cov))
        if min_eig < 1e-8:
            cov += np.eye(cov.shape[0]) * (1e-6 - min_eig)
        return cov
    
    def volatility_scale(self, weights, Sigma, target_vol=0.12):
        """Scale portfolio to target volatility"""
        port_vol = np.sqrt(weights.T @ Sigma @ weights) * np.sqrt(252)
        self.current_volatility = port_vol
        
        if port_vol > target_vol * 1.8:
            scale = (target_vol / port_vol) 
            return weights * scale + (1 - scale) / len(weights)
        return weights
    
    def optimize_weights(self, mu, Sigma, prev_weights=None, profile_constraints=None):
        import numpy as np
        import cvxpy as cp

        # --------- GET PROFILE CONSTRAINTS ---------
        if profile_constraints is None:
            # Use profile from config
            profile_constraints = get_profile_constraints(self.config.risk_profile)
    
        # --------- prep ---------
        mu_vec = mu.values if hasattr(mu, "values") else np.asarray(mu, dtype=float)
        mu_vec = np.nan_to_num(mu_vec, nan=0.0, posinf=0.0, neginf=0.0)
        n = len(mu_vec)
    
        Sigma = np.asarray(Sigma, dtype=float)
    
        pc = profile_constraints or {}
    
        # --------- profile: transform mu (multiplier + signed power) ---------
        multiplier = float(pc.get("expected_return_multiplier", 1.0))
        power = float(pc.get("multiplier_power", 1.0))
    
        mu_scaled = mu_vec * multiplier
        mu_transformed = np.sign(mu_scaled) * (np.abs(mu_scaled) + 1e-12) ** power
    
        # --------- profile: covariance shrinkage ---------
        shrinkage = float(pc.get("shrinkage", 0.0))  # 0..1
        if shrinkage > 0:
            diag = np.diag(np.diag(Sigma))
            Sigma = (1.0 - shrinkage) * Sigma + shrinkage * diag
    
        # --------- decision variable ---------
        w = cp.Variable(n)
    
        # core terms
        ret = mu_transformed @ w
        risk_aversion = float(pc.get("risk_aversion", self.config.risk_aversion))
        risk = cp.quad_form(w, Sigma)
    
        # --------- turnover + costs ---------
        cost_per_turnover = float(self.config.commission_rate + self.config.transaction_slippage)
        tc_penalty = 0.0
        turnover = None
        if prev_weights is not None:
            prev_w = np.asarray(prev_weights, dtype=float).reshape(-1)
            turnover = cp.norm1(w - prev_w)
            tc_penalty = cost_per_turnover * turnover
    
        # --------- convex "entropy-ish" smoothing (diversification) ---------
        # entropy is non-linear and tricky; this is a convex, stable proxy
        entropy_penalty = float(pc.get("entropy_penalty", 0.0))
        u = np.ones(n) / n
        div_penalty = entropy_penalty * cp.sum_squares(w - u)  # convex
    
        # --------- constraints ---------
        max_pos = float(pc.get("max_weight", self.config.max_position))
        constraints = [
            cp.sum(w) == 1,
            w >= 0,
            w <= max_pos,
        ]
    
        # optional turnover cap
        if prev_weights is not None and pc.get("max_turnover") is not None:
            constraints.append(turnover <= float(pc["max_turnover"]))
    
        # --------- objective ---------
        objective = cp.Maximize(ret - risk_aversion * risk - tc_penalty - div_penalty)
    
        # --------- solve ---------
        try:
            prob = cp.Problem(objective, constraints)
            prob.solve(solver=cp.SCS, verbose=False)
    
            if w.value is None:
                return np.ones(n) / n, {"status": "failed", "turnover": 0.0}
    
            weights_opt = np.asarray(w.value, dtype=float).reshape(-1)
            weights_opt = np.clip(weights_opt, 0.0, max_pos)
            s = weights_opt.sum()
            if s <= 0:
                return np.ones(n) / n, {"status": "failed", "turnover": 0.0}
            weights_opt /= s
    
            # target-vol scaling (your existing function)
            weights_opt = self.volatility_scale(weights_opt, Sigma, self.config.target_volatility)
            weights_opt = np.clip(weights_opt, 0.0, max_pos)
            weights_opt /= weights_opt.sum()
    
            # tiny positions post-filter (still post, because convex version requires MIQP)
            m = float(getattr(self.config, "min_trade_size", 0.0) or 0.0)
            if m > 0:
                weights_opt[weights_opt < m] = 0.0
                if weights_opt.sum() > 0:
                    weights_opt /= weights_opt.sum()
    
            # turnover report
            turn = float(np.sum(np.abs(weights_opt - prev_w))) if prev_weights is not None else 0.0
            return weights_opt, {"status": "success", "turnover": turn}
    
        except Exception:
            return np.ones(n) / n, {"status": "error", "turnover": 0.0}

    
    def ensemble_optimize(self, lstm_mu, Sigma, prev_weights=None):
        w_lstm, _ = self.optimize_weights(lstm_mu, Sigma, prev_weights)
        w_minvar = self.min_variance(Sigma)
        
        w_ensemble = (self.config.ensemble_lstm_weight * w_lstm + 
                     (1 - self.config.ensemble_lstm_weight) * w_minvar)
        
        return w_ensemble / w_ensemble.sum(), {'status': 'ensemble', 'turnover': 0}
    
    def min_variance(self, Sigma):
        n = Sigma.shape[0]
        w = cp.Variable(n)
        cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)),
                  [cp.sum(w) == 1, w >= self.config.min_position, 
                   w <= self.config.max_position]).solve(solver=cp.SCS, verbose=False)
        if w.value is not None:
            weights = np.array(w.value).flatten()
            return weights / weights.sum()
        return np.ones(n) / n


class BenchmarkStrategies:
    @staticmethod
    def equal_weight(n):
        return np.ones(n) / n
    
    @staticmethod
    def minimum_variance(Sigma, min_weight=0.0, max_weight=0.15):
        n = Sigma.shape[0]
        w = cp.Variable(n)
        try:
            cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)),
                      [cp.sum(w) == 1, w >= min_weight, w <= max_weight]).solve(solver=cp.SCS, verbose=False)
            if w.value is not None:
                weights = np.array(w.value).flatten()
                return weights / weights.sum()
        except:
            pass
        return np.ones(n) / n
    
    @staticmethod
    def risk_parity(Sigma):
        vols = np.sqrt(np.diag(Sigma))
        weights = 1.0 / (vols + 1e-8)
        return weights / weights.sum()


class PerformanceAnalyzer:
    """Performance metrics calculation"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        
    def calculate_metrics(self, returns: pd.Series):
        """
        Standard metrics assuming `returns` are DAILY returns.
        Used for your normal backtest series (daily NAV).
        """
        rf = self.config.risk_free_daily
        
        total_ret = (1 + returns).prod() - 1
        ann_ret = (1 + returns.mean()) ** 252 - 1
        ann_vol = returns.std() * np.sqrt(252)
        sharpe = (returns.mean() - rf) / (returns.std() + 1e-9) * np.sqrt(252)
        sortino = (returns.mean() - rf) / (returns[returns < 0].std() + 1e-9) * np.sqrt(252)
        
        cum = (1 + returns).cumprod()
        dd = (cum - cum.expanding().max()) / cum.expanding().max()
        max_dd = dd.min()
        
        wins = (returns > 0).sum()
        losses = (returns < 0).sum()
        win_rate = wins / (wins + losses) if (wins + losses) > 0 else 0
        calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
        
        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean()
        omega = self._omega_ratio(returns, rf)
        
        return {
            'total_return': total_ret,
            'ann_return': ann_ret,
            'ann_vol': ann_vol,
            'sharpe': sharpe,
            'sortino': sortino,
            'max_dd': max_dd,
            'calmar': calmar,
            'win_rate': win_rate,
            'var_95': var_95,
            'cvar_95': cvar_95,
            'omega': omega,
            'num_trades': len(returns)
        }

    @staticmethod
    def compute_metrics_from_nav(
        nav_series: pd.Series,
        period_length: int = 1,           # how many trading days between NAV points
        risk_free_annual: float = 0.05,
    ):
        """
        Metrics when you only have a NAV series, and each step is a multi-day period
        (e.g. one point per rebalance window).

        Example for your after-tax series:
            period_length = config.rebalance_freq  (≈ 10 trading days)
            risk_free_annual = config.risk_free_annual
        """
        if nav_series is None or len(nav_series) < 2:
            return {
                'total_return': 0.0,
                'ann_return': 0.0,
                'ann_vol': 0.0,
                'sharpe': 0.0,
                'sortino': 0.0,
                'max_dd': 0.0,
                'calmar': 0.0,
                'win_rate': 0.0,
                'var_95': 0.0,
                'cvar_95': 0.0,
                'omega': 0.0,
                'num_trades': 0,
            }

        returns = nav_series.pct_change().dropna()
        if returns.empty:
            return {
                'total_return': 0.0,
                'ann_return': 0.0,
                'ann_vol': 0.0,
                'sharpe': 0.0,
                'sortino': 0.0,
                'max_dd': 0.0,
                'calmar': 0.0,
                'win_rate': 0.0,
                'var_95': 0.0,
                'cvar_95': 0.0,
                'omega': 0.0,
                'num_trades': 0,
            }

        # number of rebalance-periods in a year
        periods_per_year = 252.0 / float(period_length)

        # risk-free per period
        rf_daily = risk_free_annual / 252.0
        rf_period = rf_daily * period_length

        total_ret = nav_series.iloc[-1] / nav_series.iloc[0] - 1.0
        mean_ret = returns.mean()
        vol = returns.std()

        ann_ret = (1.0 + mean_ret) ** periods_per_year - 1.0
        ann_vol = vol * np.sqrt(periods_per_year)

        sharpe = (mean_ret - rf_period) / (vol + 1e-9) * np.sqrt(periods_per_year)
        downside = returns[returns < 0].std()
        sortino = (mean_ret - rf_period) / (downside + 1e-9) * np.sqrt(periods_per_year)

        cum = (1.0 + returns).cumprod()
        dd = (cum - cum.expanding().max()) / cum.expanding().max()
        max_dd = dd.min()

        wins = (returns > 0).sum()
        losses = (returns < 0).sum()
        win_rate = wins / (wins + losses) if (wins + losses) > 0 else 0.0

        calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0.0

        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean()

        # simple omega (threshold 0)
        excess = returns
        gains = excess[excess > 0].sum()
        losses_sum = -excess[excess < 0].sum()
        omega = gains / losses_sum if losses_sum != 0 else np.inf

        return {
            'total_return': total_ret,
            'ann_return': ann_ret,
            'ann_vol': ann_vol,
            'sharpe': sharpe,
            'sortino': sortino,
            'max_dd': max_dd,
            'calmar': calmar,
            'win_rate': win_rate,
            'var_95': var_95,
            'cvar_95': cvar_95,
            'omega': omega,
            'num_trades': len(returns),
        }
    
    def _omega_ratio(self, returns, threshold=0.0):
        excess = returns - threshold
        gains = excess[excess > 0].sum()
        losses = -excess[excess < 0].sum()
        return gains / losses if losses != 0 else np.inf
    
    def information_ratio(self, strategy_returns, benchmark_returns):
        excess_returns = strategy_returns - benchmark_returns
        tracking_error = excess_returns.std() * np.sqrt(252)
        if tracking_error == 0:
            return 0
        return (excess_returns.mean() * 252) / tracking_error
    
    def bootstrap_confidence_interval(self, returns, metric='sharpe', n_bootstrap=1000, alpha=0.05):
        metric_samples = []
        rf = self.config.risk_free_daily
        
        for _ in range(n_bootstrap):
            sample = returns.sample(len(returns), replace=True)
            if metric == 'sharpe':
                value = (sample.mean() - rf) / sample.std() * np.sqrt(252)
            elif metric == 'return':
                value = (1 + sample.mean()) ** 252 - 1
            elif metric == 'vol':
                value = sample.std() * np.sqrt(252)
            metric_samples.append(value)
        
        ci_lower = np.percentile(metric_samples, alpha/2 * 100)
        ci_upper = np.percentile(metric_samples, (1-alpha/2) * 100)
        return ci_lower, ci_upper



class StatisticalTester:
    """Statistical significance testing"""
    
    @staticmethod
    def paired_ttest(returns1, returns2, strategy1_name="Strategy A", strategy2_name="Strategy B"):
        t_stat, p_value = stats.ttest_rel(returns1, returns2)
        
        return {
            'strategy1': strategy1_name,
            'strategy2': strategy2_name,
            't_statistic': t_stat,
            'p_value': p_value,
            'significant_95': p_value < 0.05,
            'significant_99': p_value < 0.01,
            'mean_diff': (returns1.mean() - returns2.mean()) * 252,
            'conclusion': 'Significantly different' if p_value < 0.05 else 'Not significant'
        }
    
    @staticmethod
    def sharpe_ratio_test(returns1, returns2, rf=0.05/252):
        n = len(returns1)
        
        # Work with excess DAILY returns
        ex1 = returns1 - rf
        ex2 = returns2 - rf
    
        # Daily Sharpe ratios
        sharpe1_daily = ex1.mean() / ex1.std()
        sharpe2_daily = ex2.mean() / ex2.std()
        
        # Correlation of excess returns
        rho = ex1.corr(ex2)
        
        # JK–Memmel standard error on DAILY Sharpe
        se = np.sqrt(
            (1 / n) * (
                2 - 2*rho
                + 0.5*(sharpe1_daily**2 + sharpe2_daily**2)
                - rho*(sharpe1_daily*sharpe2_daily)
            )
        )
        
        # z-statistic (no extra sqrt(252) here)
        z_stat = (sharpe1_daily - sharpe2_daily) / se
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        # For reporting: annualized Sharpe (same as before)
        sharpe1 = sharpe1_daily * np.sqrt(252)
        sharpe2 = sharpe2_daily * np.sqrt(252)
        
        return {
            'sharpe1': sharpe1,
            'sharpe2': sharpe2,
            'difference': sharpe1 - sharpe2,
            'z_statistic': z_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }


class BacktestEngine:
    """Core backtesting engine"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        self.data_manager = DataManager(config)
        self.lstm_predictor = LSTMPredictor(config)
        self.optimizer = PortfolioOptimizer(config)
        self.perf = PerformanceAnalyzer(config)
        self.stat_tester = StatisticalTester()
        
    def run_backtest(self, price_df, features, labels, test_start_date=None, test_end_date=None, period_name="Full"):
        """Run backtest with app features"""
        
        print(f"\n{'='*70}")
        print(f"BACKTEST: {period_name}")
        print(f"{'='*70}")
        
        
        
        dates = sorted(features.index.unique())

        # Find test start index
        if test_start_date:
            test_dates = [d for d in dates if d >= pd.Timestamp(test_start_date)]
            if not test_dates:
                print(f"⚠️  No data after {test_start_date}")
                return None
            start_idx = dates.index(test_dates[0])
        else:
            start_idx = self.config.train_window_days + self.config.validation_window
        
        # Find test end index
        if test_end_date:
            test_end_dates = [d for d in dates if d <= pd.Timestamp(test_end_date)]
            if test_end_dates:
                end_idx = dates.index(test_end_dates[-1])
            else:
                end_idx = len(dates) - max(self.config.pred_horizons) - 1
        else:
            end_idx = len(dates) - max(self.config.pred_horizons) - 1
        
        if start_idx >= len(dates):
            print(f"⚠️  Not enough history: need {start_idx} days, have {len(dates)}")
            return None
            
        if start_idx >= end_idx:
            print(f"⚠️  Insufficient data: start_idx={start_idx}, end_idx={end_idx}")
            return None
        
        print(f"Period: {dates[start_idx].date()} to {dates[end_idx].date()}")
        print(f"Trading days: {end_idx - start_idx}")
        print(f"Assets: {len(self.config.tickers)}\n")
        
        strategy_keys = [f"LSTM_{h}day" for h in self.config.pred_horizons]
        if self.config.enable_ensemble:
            strategy_keys.append('Ensemble_LSTM_MinVar')
        strategy_keys.extend(['Equal_Weight', 'Min_Variance', 'Risk_Parity'])
        
        navs = {k: 1.0 for k in strategy_keys}
        nav_series = {k: [] for k in strategy_keys}
        nav_dates = []
        prev_weights = {k: None for k in strategy_keys}
        weights_history = {k: [] for k in strategy_keys}
        prev_mu_signal = {k: None for k in strategy_keys}  # NEW: store mu at last rebalance per strategy
        
        crisis_mode_count = 0
        rebalance_skipped = 0
        
        pbar = tqdm(range(start_idx, end_idx, self.config.rebalance_freq),
                   desc=f"Backtesting {period_name}", ncols=100)
        
        for i in pbar:
            train_end = i - self.config.validation_window
            val_end = i
            test_date = dates[i]
            
            train_dates = dates[train_end - self.config.train_window_days:train_end]
            val_dates = dates[train_end:val_end]
            
            # Crisis detection
            # Crisis detection - INITIALIZE crisis_scale FIRST
            crisis_scale = 1.0  # ← Move this OUTSIDE the if block
            
            recent_ret = price_df.pct_change().loc[train_dates[-20:], self.config.tickers].dropna()
            if len(recent_ret) > 5:
                recent_vol = recent_ret.std().mean()
            
                if recent_vol > self.config.crisis_vol_threshold:
                    crisis_mode_count += 1
                    self.optimizer.crisis_mode = True
                    
                    # Adjust defensive posture based on risk profile
                    if self.config.risk_profile == "aggressive":
                        crisis_scale = 0.8  # Only 20% defensive for aggressive profile
                    else:
                        crisis_scale = 0.5  # 50% defensive for conservative profile
                    
                    if self.config.verbose:
                        print(f"\n  ⚠️ CRISIS MODE at {test_date.date()} (vol={recent_vol:.4f})")
                    pbar.set_description(f"{period_name}: {test_date.date()} [CRISIS]")
                else:
                    self.optimizer.crisis_mode = False
                    crisis_scale = 1.0
            else:
                # Not enough data for crisis detection
                self.optimizer.crisis_mode = False
                crisis_scale = 1.0  # Explicitly set to avoid NameError
            
            pbar.set_description(f"{period_name}: {test_date.date()}")
            
            # Train/retrain
            if i == start_idx or i % self.config.retrain_cadence == 0:
                if self.config.verbose:
                    print(f"\n  Training at {test_date.date()} ({len(self.config.tickers)} assets)...")
                
                for ticker in self.config.tickers:
                    fcols = [c for c in features.columns if c[0] == ticker and c[1] not in ['price', 'ret']]
                    X_train = features.loc[train_dates, fcols].copy().dropna()
                    X_val = features.loc[val_dates, fcols].copy().dropna()
                    
                    if len(X_train) < self.config.lstm_lookback + 50:
                        continue
                    
                    for h in self.config.pred_horizons:
                        y_train = labels[h].loc[X_train.index, ticker].fillna(0).values
                        y_val = labels[h].loc[X_val.index, ticker].fillna(0).values if len(X_val) > 0 else None
                        
                        self.lstm_predictor.train_model(
                            ticker, h, X_train.values, y_train,
                            X_val.values if len(X_val) > 0 else None, y_val
                        )
            
            # Predictions
            predictions = {h: {} for h in self.config.pred_horizons}
            
            for ticker in self.config.tickers:
                fcols = [c for c in features.columns if c[0] == ticker and c[1] not in ['price', 'ret']]
                X_recent = features.loc[train_dates[-self.config.lstm_lookback:], fcols].copy()
                
                if len(X_recent) >= self.config.lstm_lookback:
                    for h in self.config.pred_horizons:
                        predictions[h][ticker] = self.lstm_predictor.predict(ticker, h, X_recent.values)
                else:
                    for h in self.config.pred_horizons:
                        predictions[h][ticker] = 0.0
            
            # Store predictions in optimizer
            self.optimizer.predictions = predictions[self.config.pred_horizons[0]]
            
            # Scale predictions MORE AGGRESSIVELY
            # Scale predictions MORE AGGRESSIVELY
            for h in self.config.pred_horizons:
                pred_values = list(predictions[h].values())
                if len(pred_values) > 0:
                    # Clean predictions first - remove NaN/Inf
                    clean_values = [v for v in pred_values if np.isfinite(v)]
                    
                    if len(clean_values) == 0:  # All predictions are invalid
                        # Set all to zero
                        for ticker in predictions[h]:
                            predictions[h][ticker] = 0.0
                        continue
                    
                    pred_mean = np.mean(clean_values)
                    pred_std = np.std(clean_values)
                    
                    # If std is too small, skip normalization
                    if pred_std < 1e-6:
                        for ticker in predictions[h]:
                            predictions[h][ticker] = 0.0
                        continue
                    
                    # Z-score normalization to preserve relative differences
                    for ticker in predictions[h]:
                        pred_val = predictions[h][ticker]
                        
                        # Replace NaN/Inf with 0
                        if not np.isfinite(pred_val):
                            predictions[h][ticker] = 0.0
                            continue
                        
                        z_score = (pred_val - pred_mean) / pred_std
                        
                        # Amplify signal in aggressive mode
                        if self.config.risk_profile == "aggressive":
                            z_score *= 1.5
                        
                        # Apply tanh to bound but preserve differences
                        predictions[h][ticker] = np.tanh(z_score) * 0.25
                        
                        # Final safety check
                        if not np.isfinite(predictions[h][ticker]):
                            predictions[h][ticker] = 0.0
            
            # ← ADD THE NEW MOMENTUM CODE HERE (right after the for loop ends)
            # Add momentum boost for aggressive profile
            # Add momentum boost for aggressive profile
            if self.config.risk_profile == "aggressive":
                # Calculate recent momentum (last 20 days)
                if len(train_dates) >= 20:
                    try:
                        recent_prices = price_df.loc[train_dates[-20:], self.config.tickers]
                        if len(recent_prices) >= 20:
                            # Calculate momentum with NaN handling
                            momentum = (recent_prices.iloc[-1] / recent_prices.iloc[0] - 1)
                            momentum = momentum.fillna(0.0)  # ← FIX: Replace NaN with 0
                            
                            for h in self.config.pred_horizons:
                                for ticker in predictions[h]:
                                    if ticker in momentum.index:
                                        mom_value = momentum[ticker]
                                        # Additional safety: check if mom_value is valid number
                                        if pd.notna(mom_value) and np.isfinite(mom_value):  # ← FIX: Validate
                                            if mom_value > 0.05:
                                                predictions[h][ticker] *= 1.2
                                            elif mom_value < -0.05:
                                                predictions[h][ticker] *= 0.8
                    except Exception as e:
                        # Fail silently - momentum boost is optional
                        if self.config.verbose:
                            print(f"  ⚠️ Momentum calculation failed: {e}")
                        pass

            

            
            # Covariance
            recent_ret_cov = price_df.pct_change().loc[train_dates[-self.config.ewma_span:], 
                                                       self.config.tickers].dropna()
            if len(recent_ret_cov) < 20:
                continue
            
            try:
                Sigma = self.optimizer.ewma_covariance(recent_ret_cov)
            except:
                continue
            
            opt_info = {}
            
            # Optimize weights
            for h in self.config.pred_horizons:
                key = f"LSTM_{h}day"
                mu = pd.Series({t: predictions[h].get(t, 0.0) for t in self.config.tickers})
                
                # NEW: Check if rebalancing needed
                weights_dict = {self.config.tickers[i]: prev_weights[key][i] 
                              for i in range(len(self.config.tickers))} if prev_weights[key] is not None else None
                
                
                mu_dict = mu.to_dict()

                if prev_weights[key] is not None and (not self.optimizer.should_rebalance_signal(prev_mu_signal[key], mu_dict)):
                    rebalance_skipped += 1
                    if self.config.verbose and rebalance_skipped % 5 == 0:
                        print(f"  Skipped {rebalance_skipped} rebalances (signal change below threshold)")
                    w = prev_weights[key]
                    info = {'turnover': 0, 'status': 'no_rebalance'}
                else:
                    w, info = self.optimizer.optimize_weights(mu, Sigma, prev_weights.get(key))
                    prev_weights[key] = w
                    prev_mu_signal[key] = mu_dict  # store mu at the last rebalance


                self.optimizer.current_weights = {
                    self.config.tickers[i]: w[i] for i in range(len(w))
                }
                opt_info[key] = info
                weights_history[key].append({'date': test_date, 'weights': w.copy()})

                
                if self.config.enable_ensemble:
                    w_ensemble, info_ens = self.optimizer.ensemble_optimize(mu, Sigma, prev_weights.get('Ensemble_LSTM_MinVar'))
                    prev_weights['Ensemble_LSTM_MinVar'] = w_ensemble
                    opt_info['Ensemble_LSTM_MinVar'] = info_ens
                    weights_history['Ensemble_LSTM_MinVar'].append({'date': test_date, 'weights': w_ensemble.copy()})
            
            # Benchmarks (assigned AFTER crisis scaling, so they remain unaffected)
            equal_w = BenchmarkStrategies.equal_weight(len(self.config.tickers))
            minvar_w = BenchmarkStrategies.minimum_variance(Sigma, self.config.min_position, self.config.max_position)
            riskpar_w = BenchmarkStrategies.risk_parity(Sigma)

            # Benchmarks
            # Apply crisis mode scaling BEFORE benchmark assignment
            if crisis_scale < 1.0:
                for h in self.config.pred_horizons:
                    key = f"LSTM_{h}day"
                    if prev_weights.get(key) is not None:
                        w = prev_weights[key]
                        w = crisis_scale * w + (1 - crisis_scale) * minvar_w
                        w = np.maximum(w, 0)
                        w = w / (w.sum() + 1e-12)
                        prev_weights[key] = w
            
                if self.config.enable_ensemble and prev_weights.get('Ensemble_LSTM_MinVar') is not None:
                    w = prev_weights['Ensemble_LSTM_MinVar']
                    w = crisis_scale * w + (1 - crisis_scale) * minvar_w
                    w = np.maximum(w, 0)
                    w = w / (w.sum() + 1e-12)
                    prev_weights['Ensemble_LSTM_MinVar'] = w
            
                        
            prev_weights['Equal_Weight'] = equal_w
            prev_weights['Min_Variance'] = minvar_w
            prev_weights['Risk_Parity'] = riskpar_w
            
            for bench_key, bench_w in [('Equal_Weight', equal_w), ('Min_Variance', minvar_w), ('Risk_Parity', riskpar_w)]:
                weights_history[bench_key].append({'date': test_date, 'weights': bench_w.copy()})
                opt_info[bench_key] = {'turnover': 0}
            
            # Apply weights and calculate returns
            apply_start = i + 1
            apply_end = min(i + self.config.rebalance_freq, len(dates) - 1)
            apply_dates = [d for d in dates[apply_start:apply_end+1] if d in price_df.index]
            
            for ad in apply_dates:
                try:
                    ad_idx = price_df.index.get_loc(ad)
                    if ad_idx == 0:
                        continue
                    
                    prev_date = price_df.index[ad_idx - 1]
                    daily_ret = ((price_df.loc[ad, self.config.tickers] - 
                                price_df.loc[prev_date, self.config.tickers]) / 
                                price_df.loc[prev_date, self.config.tickers])
                    daily_ret = daily_ret.fillna(0)
                    
                    for key in strategy_keys:
                        w = prev_weights[key] if prev_weights[key] is not None else equal_w
                        port_ret = float(np.dot(w, daily_ret.values))
                        
                        # Apply transaction costs
                        if ad == test_date and key in opt_info:
                            total_cost = (self.config.commission_rate + self.config.transaction_slippage) * opt_info[key].get('turnover', 0)
                            port_ret -= total_cost
                        
                        navs[key] *= (1 + port_ret)
                        nav_series[key].append(navs[key])
                        
                        # Update drawdown
                        if key == f"LSTM_{self.config.pred_horizons[0]}day":
                            peak = max(nav_series[key])
                            self.optimizer.current_drawdown = (navs[key] - peak) / peak
                    
                    nav_dates.append(ad)
                except:
                    continue
            
            if len(nav_series[strategy_keys[0]]) > 0:
                pbar.set_postfix({'NAV': f'{nav_series[strategy_keys[0]][-1]:.3f}'})
        
        pbar.close()
        
        if crisis_mode_count > 0:
            print(f"\n⚠️ Crisis mode triggered {crisis_mode_count} times")
        if rebalance_skipped > 0:
            print(f"💰 Saved {rebalance_skipped} rebalances (below {self.config.rebalance_threshold*100:.0f}% threshold)")
        
        results = {'navs': {}, 'metrics': {}, 'weights_history': weights_history}
        
        # Calculate metrics
        for key in strategy_keys:
            if not nav_series[key]:
                continue
            
            series = pd.Series(nav_series[key], index=nav_dates[:len(nav_series[key])])
            results['navs'][key] = series
            
            returns = series.pct_change().dropna()
            if len(returns) > 10:
                results['metrics'][key] = self.perf.calculate_metrics(returns)
        
        # NEW: Save live weights for app
        weights_file = os.path.join(self.config.output_dir, "live_weights", f"{period_name}_latest_weights.json")
        with open(weights_file, 'w') as f:
            json.dump(self.optimizer.get_current_portfolio_weights(), f, indent=2)
        
        # NEW: Save attribution analysis
        if 'LSTM_5day' in weights_history and len(weights_history['LSTM_5day']) > 0:
            price_returns = price_df.pct_change()
            weights_dict_list = [dict(zip(self.config.tickers, w['weights'])) 
                               for w in weights_history['LSTM_5day']]
            attribution = self.optimizer.analyze_attribution(price_returns, weights_dict_list)
            
            attr_file = os.path.join(self.config.output_dir, "live_weights", f"{period_name}_attribution.json")
            with open(attr_file, 'w') as f:
                json.dump(attribution, f, indent=2, default=str)
        
        return results


class OutOfSampleValidator:
    """Out-of-sample validation"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        self.backtest_engine = BacktestEngine(config)
        
    def run_walk_forward_validation(self):
        
        print("\n" + "="*70)
        print("PRODUCTION-READY MULTI-ASSET PORTFOLIO SYSTEM")
        print("="*70)
        print(f"Testing {len(self.config.out_of_sample_periods)} periods")
        print(f"Universe: {len(self.config.tickers)} assets")
        print(f"\nApp Features Enabled:")
        print(f"  • Transaction costs: {(self.config.commission_rate + self.config.transaction_slippage)*10000:.1f} bps")
        print(f"  • Rebalance threshold: {self.config.rebalance_threshold*100:.0f}%")
        print(f"  • Min trade size: {self.config.min_trade_size*100:.0f}%")
        print(f"  • Crisis detection: {self.config.crisis_vol_threshold*100:.0f}% daily vol")
        print(f"  • Asset health checks: Enabled")
        print(f"  • Live weights export: Enabled\n")
        
        price_df = self.backtest_engine.data_manager.download_data()
        features = self.backtest_engine.data_manager.compute_features(price_df)
        labels = self.backtest_engine.data_manager.create_labels(price_df)
        
        if len(price_df) == 0:
            print("❌ No data downloaded")
            return {}
        
        print(f"Full data: {price_df.index[0].date()} to {price_df.index[-1].date()}")
        print(f"Total days: {len(price_df)}\n")
        
        all_results = {}
        
        for period in self.config.out_of_sample_periods:
            print(f"\n{'='*70}")
            print(f"PERIOD: {period['name']}")
            print(f"{'='*70}")
            print(f"Train: {period['train_start']} → {period['train_end']}")
            print(f"Test:  {period['test_start']} → {period['test_end']}")
            print("="*70)
            
            train_data = price_df.loc[period['train_start']:period['train_end']]
            test_data = price_df.loc[period['test_start']:period['test_end']]
            
            print(f"\nTrain days: {len(train_data)}")
            print(f"Test days:  {len(test_data)}")
            
            if len(train_data) < 300:
                print("⚠️  Insufficient training data, skipping")
                continue
            
            if len(test_data) < 60:
                print("⚠️  Insufficient test data, skipping")
                continue
            
            self.backtest_engine.lstm_predictor = LSTMPredictor(self.config)
            self.backtest_engine.optimizer = PortfolioOptimizer(self.config)
            
            combined_data = price_df.loc[period['train_start']:period['test_end']]
            combined_features = features.loc[period['train_start']:period['test_end']]
            
            results = self.backtest_engine.run_backtest(
                combined_data, combined_features, labels,
                test_start_date=period['test_start'],
                test_end_date=period['test_end'],
                period_name=period['name']
            )
            
            if results:
                all_results[period['name']] = results
                
                period_dir = os.path.join(self.config.output_dir, "out_of_sample", period['name'])
                os.makedirs(period_dir, exist_ok=True)
                
                for key, series in results['navs'].items():
                    series.to_csv(os.path.join(period_dir, f"nav_{key}.csv"))
                
                if results['metrics']:
                    pd.DataFrame(results['metrics']).T.to_csv(
                        os.path.join(period_dir, "metrics.csv"))
        
        return all_results
    
    def compare_periods(self, all_results):
        
        print("\n" + "="*70)
        print("OUT-OF-SAMPLE PERFORMANCE COMPARISON")
        print("="*70 + "\n")
        
        comparison_data = {}
        
        for period_name, results in all_results.items():
            if 'metrics' in results and results['metrics']:
                lstm_key = f"LSTM_{self.config.pred_horizons[0]}day"
                if lstm_key in results['metrics']:
                    comparison_data[period_name] = results['metrics'][lstm_key]
        
        if comparison_data:
            df = pd.DataFrame(comparison_data).T
            print("LSTM Performance Across Periods:")
            print("-" * 70)
            print(df[['sharpe', 'ann_return', 'ann_vol', 'max_dd', 'calmar']])
            
            print("\n\nAverage Out-of-Sample Performance:")
            print("-" * 70)
            print(df[['sharpe', 'ann_return', 'ann_vol', 'max_dd']].mean())
            
            df.to_csv(os.path.join(self.config.output_dir, "out_of_sample", "period_comparison.csv"))
        
        return comparison_data


class ResearchVisualizer:
    """Visualization"""
    
    def __init__(self, config: ResearchConfig):
        self.config = config
        
    def plot_out_of_sample_comparison(self, all_results):
        
        fig, axes = plt.subplots(len(all_results), 1, figsize=(16, 6*len(all_results)), sharex=False)
        
        if len(all_results) == 1:
            axes = [axes]
        
        for ax, (period_name, results) in zip(axes, all_results.items()):
            if 'navs' in results:
                for key, series in results['navs'].items():
                    alpha = 1.0 if 'LSTM' in key or 'Ensemble' in key else 0.6
                    linewidth = 2.5 if 'LSTM' in key or 'Ensemble' in key else 1.5
                    ax.plot(series.index, series.values, label=key, alpha=alpha, linewidth=linewidth)
                
                ax.set_title(f"{period_name} - NAV Comparison", fontsize=14, fontweight='bold')
                ax.set_ylabel("NAV")
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
                ax.grid(True, alpha=0.3)
        
        plt.xlabel("Date")
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_dir, "plots", "out_of_sample_nav_comparison.png"),
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Plots saved")
    # ← ADD THE NEW METHOD HERE ↓↓↓
    def plot_allocation_over_time(self, weights_history, period_name="Strategy"):
        """Plot how asset allocation evolved over time"""
        
        if not weights_history or len(weights_history) == 0:
            print(f"No weight history for {period_name}")
            return
        
        # Extract dates and weights
        dates = [entry['date'] for entry in weights_history]
        tickers = self.config.tickers
        
        # Build weight matrix (time x assets)
        weight_matrix = np.zeros((len(dates), len(tickers)))
        for i, entry in enumerate(weights_history):
            weight_matrix[i, :] = entry['weights']
        
        # Create DataFrame
        df_weights = pd.DataFrame(weight_matrix, index=dates, columns=tickers)
        
        # Group by asset class for cleaner visualization
        asset_classes = {
            'Tech': ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA'],
            'Financial': ['V', 'JPM'],
            'Consumer': ['WMT'],
            'India': ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ITC'],
            'ETFs': ['SPY', 'QQQ', 'IWM', 'EFA', 'VTI'],
            'Bonds': ['AGG', 'TLT', 'IEF', 'LQD', 'HYG'],
            'REITs': ['VNQ', 'O', 'PLD'],
            'Commodities': ['GLD', 'SLV', 'USO', 'DBA'],
            'Sectors': ['XLK', 'XLF', 'XLV', 'XLE'],
            'Dividend': ['VYM', 'SCHD', 'DVY', 'VTV']
        }
        
        # Aggregate by asset class
        class_weights = {}
        for class_name, tickers_list in asset_classes.items():
            class_cols = [t for t in tickers_list if t in df_weights.columns]
            if class_cols:
                class_weights[class_name] = df_weights[class_cols].sum(axis=1)
        
        df_classes = pd.DataFrame(class_weights, index=dates)
        
        # Create stacked area plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
        
        # Plot 1: Asset class allocation over time
        df_classes.plot.area(ax=ax1, alpha=0.7, linewidth=0.5)
        ax1.set_title(f'{period_name} - Asset Class Allocation Over Time', 
                     fontsize=14, fontweight='bold')
        ax1.set_ylabel('Weight (%)', fontsize=12)
        ax1.set_xlabel('')
        ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim([0, 1])
        
        # Plot 2: Top 10 individual holdings over time
        top10_cols = df_weights.iloc[-1].nlargest(10).index
        df_top10 = df_weights[top10_cols]
        
        for col in top10_cols:
            ax2.plot(dates, df_top10[col], label=col, linewidth=2, alpha=0.8)
        
        ax2.set_title(f'{period_name} - Top 10 Holdings Over Time', 
                     fontsize=14, fontweight='bold')
        ax2.set_ylabel('Weight (%)', fontsize=12)
        ax2.set_xlabel('Date', fontsize=12)
        ax2.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save plot
        filename = f"allocation_over_time_{period_name}.png"
        plt.savefig(os.path.join(self.config.output_dir, "plots", filename),
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Saved allocation plot: {filename}")
    # ← END OF NEW METHOD ↑↑↑


2025-12-24 17:51:44.701777: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766598704.900000      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766598704.956212      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766598705.431201      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766598705.431253      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766598705.431256      23 computation_placer.cc:177] computation placer alr

✓ GPU Enabled: 1 device(s)


In [2]:
# === Tax optimization helpers (fixed & consistent) ===
import pandas as pd
import numpy as np

def compute_rebalance_realized_gains(old_weights, new_weights, portfolio_value, cost_basis):
    """
    Realized gains from rebalancing under an average-cost assumption.

    Key FIXES:
    1) Do NOT clamp cost basis to post-rebalance value (that can fabricate "free" basis changes).
    2) Compute realized gain = fraction_sold * (current_value - basis).
    3) Update cost basis:
       - Remaining basis scales down by (1 - fraction_sold)
       - New buys add basis equal to dollars spent (buy_value)
    """
    # Align all series to the same asset universe
    assets = old_weights.index.union(new_weights.index).union(cost_basis.index)
    old_w = old_weights.reindex(assets).fillna(0.0)
    new_w = new_weights.reindex(assets).fillna(0.0)
    basis = cost_basis.reindex(assets).fillna(0.0)

    # Current and target dollar values (based on weights at rebalance time)
    current_values = old_w * portfolio_value
    target_values  = new_w * portfolio_value

    # Dollar sells and buys (can both be >0 due to rounding; OK)
    sell_value = np.maximum(current_values - target_values, 0.0)
    buy_value  = np.maximum(target_values - current_values, 0.0)
    trade_volume = sell_value + buy_value

    # Fraction of each position sold (avg-cost method)
    with np.errstate(divide="ignore", invalid="ignore"):
        fraction_sold = sell_value / current_values.replace(0.0, np.nan)
    fraction_sold = fraction_sold.fillna(0.0).clip(0.0, 1.0)

    # Unrealized gain per asset before rebalance
    unrealized_gain = current_values - basis

    # Realized gain proportional to fraction sold
    realized_gains = fraction_sold * unrealized_gain

    # Update cost basis:
    # - keep remaining basis after sells
    # - add new purchases at cost = dollars spent (buy at market)
    new_basis = basis * (1.0 - fraction_sold) + buy_value

    return {
        "realized_gains": realized_gains,
        "trade_volume": trade_volume,
        "new_cost_basis": new_basis,
    }


def apply_tax_loss_harvest(realized_gains, realized_losses, tax_rate):
    """
    TLH with carry-forward losses.

    Inputs:
      realized_gains: >=0 (scalar)
      realized_losses: <=0 (scalar, includes carry_losses)
    Outputs:
      tax_due: >=0
      remaining_losses: <=0
    """
    gains_sum = float(realized_gains) if np.isscalar(realized_gains) else float(np.sum(realized_gains))
    losses_sum = float(realized_losses) if np.isscalar(realized_losses) else float(np.sum(realized_losses))

    gains_sum = max(0.0, gains_sum)
    losses_sum = min(0.0, losses_sum)

    net_taxable = gains_sum + losses_sum  # losses_sum is negative
    if net_taxable > 0:
        tax_due = net_taxable * tax_rate
        remaining_losses = 0.0
    else:
        tax_due = 0.0
        remaining_losses = net_taxable  # negative carry forward

    return {
        "tax_due": max(0.0, float(tax_due)),
        "remaining_losses": min(0.0, float(remaining_losses)),
    }


def simulate_after_tax_returns(returns, weight_history, tax_rate=0.15, harvest=True):
    """
    FIX: Make accounting consistent by tracking:
      - holdings ($ per asset)
      - cost_basis ($ per asset)

    Why this matters:
      Paying tax reduces portfolio value; if you only reduce 'pv' but not the asset holdings/basis,
      the next rebalance uses inconsistent state and can create after_tax > gross artifacts.

    Output indexed by rebalance dates with columns:
      gross: portfolio value before tax payment at each rebalance checkpoint
      after_tax: portfolio value after paying taxes at that checkpoint
    """
    returns = returns.copy()
    returns.index = pd.to_datetime(returns.index)

    dates = sorted(pd.to_datetime(list(weight_history.keys())))
    if len(dates) == 0:
        return pd.DataFrame(columns=["gross", "after_tax"])

    # Align weights to asset universe
    assets = returns.columns
    w0 = pd.Series(weight_history[dates[0]]).reindex(assets).fillna(0.0)
    w0 = w0 / max(w0.sum(), 1e-12)

    # Start with NAV=1.0 invested per w0
    holdings = w0 * 1.0          # $ holdings per asset
    cost_basis = holdings.copy() # $ cost basis per asset (average cost)
    carry_losses = 0.0

    gross_series = []
    after_tax_series = []

    for i in range(len(dates)):
        start = dates[i]
        end = dates[i + 1] if i + 1 < len(dates) else returns.index.max()

        # Apply returns from (start, end] to holdings
        period = returns.loc[start:end]
        if period.shape[0] >= 2:
            period = period.iloc[1:]  # advance forward; avoid double-counting start
        else:
            period = period.iloc[0:0]

        if period.shape[0] > 0:
            cum = (1.0 + period).prod(axis=0).reindex(assets).fillna(1.0)
            holdings = holdings * cum

        gross_portfolio = float(holdings.sum())

        if i + 1 < len(dates):
            # Rebalance at end date to next weights (using current gross_portfolio)
            next_date = dates[i + 1]
            next_w = pd.Series(weight_history[next_date]).reindex(assets).fillna(0.0)
            next_w = next_w / max(next_w.sum(), 1e-12)

            current_w = (holdings / gross_portfolio) if gross_portfolio > 0 else w0 * 0.0
            current_w = current_w.reindex(assets).fillna(0.0)

            rg = compute_rebalance_realized_gains(
                old_weights=current_w,
                new_weights=next_w,
                portfolio_value=gross_portfolio,
                cost_basis=cost_basis,
            )

            realized = rg["realized_gains"].reindex(assets).fillna(0.0)

            pos_gains = float(realized[realized > 0].sum()) if (realized > 0).any() else 0.0
            pos_losses = float(realized[realized < 0].sum()) if (realized < 0).any() else 0.0  # negative

            # Update holdings to target (post-trade, pre-tax-payment)
            holdings = next_w * gross_portfolio

            # Update cost basis consistent with those trades
            cost_basis = rg["new_cost_basis"].reindex(assets).fillna(0.0)

            # Compute tax due (never negative)
            if harvest:
                res = apply_tax_loss_harvest(pos_gains, pos_losses + carry_losses, tax_rate)
                tax_due = float(res["tax_due"])
                carry_losses = float(res["remaining_losses"])
            else:
                tax_due = float(max(0.0, pos_gains) * tax_rate)

            tax_due = max(0.0, min(tax_due, gross_portfolio))

            # Pay tax from portfolio: shrink BOTH holdings and cost basis proportionally
            if tax_due > 0 and gross_portfolio > 0:
                scale = (gross_portfolio - tax_due) / gross_portfolio
                holdings *= scale
                cost_basis *= scale

            after_tax = float(holdings.sum())

            # Hard guarantee: after_tax <= gross_portfolio (tiny epsilon allowed)
            if after_tax > gross_portfolio + 1e-10:
                after_tax = gross_portfolio

        else:
            # Final period: no rebalance, no realization => after-tax equals gross
            after_tax = gross_portfolio

        gross_series.append(gross_portfolio)
        after_tax_series.append(after_tax)

    result = pd.DataFrame({"gross": gross_series, "after_tax": after_tax_series}, index=dates[:len(gross_series)])

    # Final safety cap (should be unnecessary now)
    result["after_tax"] = np.minimum(result["after_tax"], result["gross"])

    return result


In [3]:
import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, Union
from numpy.linalg import inv

# ----------------------
# Stronger profile constraints (with power & shrinkage knobs)
# ---------------------



# ----------------------
# Enhanced optimizer with non-linear mu transformation
# ----------------------
def optimize_weights_with_profile(
    returns: pd.DataFrame,
    profile_constraints: Dict[str, Any],
    expected_returns: Optional[pd.Series] = None,
    method: str = "mean_variance",
    return_multiplier: bool = False
) -> Union[pd.Series, Tuple[pd.Series, float]]:
    """
    Mean-variance style optimizer compatible with 'method' and returning multiplier optionally.
    Key differences:
      - Applies profile_constraints['expected_return_multiplier']
      - Applies non-linear transform: mu_signed * (abs(mu_signed) ** multiplier_power)
      - Uses profile-specific shrinkage & entropy smoothing
    """
    # Safety fallback
    if returns is None or returns.shape[0] < 20:
        n = len(returns.columns) if returns is not None else 0
        weights = pd.Series(np.ones(n) / max(n, 1), index=returns.columns if returns is not None else [])
        return (weights, float(profile_constraints.get('expected_return_multiplier', 1.0))) if return_multiplier else weights

    # Annualize or use provided expected_returns
    mu = (returns.mean() * 252.0)
    if expected_returns is not None:
        mu = expected_returns.reindex(returns.columns).fillna(0.0)

    # Extract profile parameters (with sensible defaults)
    multiplier = float(profile_constraints.get('expected_return_multiplier', 1.0))
    power = float(profile_constraints.get('multiplier_power', 1.0))
    gamma = float(profile_constraints.get('risk_aversion', 2.0))
    shrinkage = float(profile_constraints.get('shrinkage', 0.12))
    entropy_penalty = float(profile_constraints.get('entropy_penalty', 1e-3))
    max_w = float(profile_constraints.get('max_weight', 1.0))

    # Apply linear multiplier first
    mu_scaled = mu * multiplier

    # Non-linear amplification/compression while preserving sign
    signed = np.sign(mu_scaled.values)
    mag = np.abs(mu_scaled.values) + 1e-12
    mag_transformed = mag ** power
    mu_transformed = signed * mag_transformed
    mu_vec = pd.Series(mu_transformed, index=mu.index)

    # Covariance (annualized) and shrinkage (profile-specific)
    cov = returns.cov() * 252.0
    lam = shrinkage
    cov_shrunk = (1 - lam) * cov + lam * np.diag(np.diag(cov))

    # Ensure invertible
    try:
        inv_cov = inv(cov_shrunk.values)
    except Exception:
        cov_shrunk += np.eye(cov_shrunk.shape[0]) * 1e-6
        inv_cov = inv(cov_shrunk.values)

    n = cov_shrunk.shape[0]

    # Method selection
    if method == "min_variance":
        ones = np.ones(n)
        raw = inv_cov.dot(ones)
    else:
        raw = inv_cov.dot(mu_vec.values) / gamma

    # No short: clip negatives
    raw = np.maximum(raw, 0.0)

    # If all zeros -> uniform
    if raw.sum() <= 0:
        w = pd.Series(np.ones_like(raw) / len(raw), index=cov_shrunk.index)
        return (w, multiplier) if return_multiplier else w

    # Normalize and apply hard cap
    w = raw / raw.sum()
    w = np.minimum(w, max_w)

    # Project to caps: iterative proportional renormalization
    def project_to_caps_vec(x, cap):
        x = np.minimum(x, cap)
        if x.sum() == 0:
            return np.ones_like(x) / len(x)
        return x / x.sum()

    w = project_to_caps_vec(w, max_w)
    w = pd.Series(w, index=cov_shrunk.index)

    # Entropy-like smoothing toward uniform, scaled by concentration
    if entropy_penalty and entropy_penalty > 0:
        uniform = pd.Series(np.ones(len(w)) / len(w), index=w.index)
        concentration = (w.max() - w.mean())
        alpha = min(0.10, entropy_penalty * (1.0 + concentration * 10.0))
        w = (1 - alpha) * w + alpha * uniform
        w = w.clip(lower=0.0)
        w = w / w.sum()

    # Final normalize safety
    if w.sum() <= 0:
        w = pd.Series(np.ones(len(w)) / len(w), index=w.index)
    else:
        w = w / w.sum()

    return (w, multiplier) if return_multiplier else w





In [4]:
def max_drawdown_from_nav(nav: pd.Series) -> float:
    nav = nav.dropna()
    if len(nav) < 2:
        return 0.0
    peak = nav.cummax()
    dd = (nav / peak) - 1.0
    return float(dd.min())

def main():
    
    print("\n" + "="*70)
    print("PRODUCTION-READY PORTFOLIO OPTIMIZATION SYSTEM")
    print("="*70)
    print("\nApp Features:")
    print("  ✅ Real transaction costs (commissions + slippage)")
    print("  ✅ Intelligent rebalancing (skip if drift < 2%)")
    print("  ✅ Asset health validation")
    print("  ✅ Crisis mode detection & protection")
    print("  ✅ Live portfolio weights export (JSON)")
    print("  ✅ Performance attribution analysis")
    print("  ✅ Risk alerts & explanations")
    print("  ✅ Dollar rounding for real accounts")
    print("  ✅ Tax-aware performance (after-tax NAV)")
    print("  ✅ Personalized risk-profile portfolios")
    print("\nReady for Research Paper & Real-Money App!\n")
    print("="*70 + "\n")
    
    # ----------------- CORE SETUP -----------------
    #choice = input("Choose risk profile ['c' = conservative (normal), 'a' = aggressive]: ").strip().lower()
    #if choice in ("a", "aggressive", "high"):
        #chosen_profile = "aggressive"
    #else:
        #chosen_profile = "conservative"
    #print(f"\nUsing risk profile: {chosen_profile.capitalize()}")

    chosen_profile="aggressive"
    print(f"\nUsing risk profile: {chosen_profile.capitalize()}")
    config = ResearchConfig()
    config.configure_for_profile(chosen_profile)
    config.create_directories()
    validator = OutOfSampleValidator(config)
    all_results = validator.run_walk_forward_validation()
    
    if not all_results:
        print("\n❌ No valid results\n")
        return
    
    comparison = validator.compare_periods(all_results)
    
    # ----------------- STATISTICAL TESTING -----------------
    print("\n" + "="*70)
    print("STATISTICAL TESTING")
    print("="*70 + "\n")
    
    stat_tester = StatisticalTester()
    perf_analyzer = PerformanceAnalyzer(config)
    lstm_key = f"LSTM_{config.pred_horizons[0]}day"
    
    for period_name, results in all_results.items():
        if "navs" not in results:
            continue
        
        print(f"\n{period_name}:")
        print("-" * 70)
        
        if lstm_key in results["navs"] and "Min_Variance" in results["navs"]:
            lstm_returns = results["navs"][lstm_key].pct_change().dropna()
            minvar_returns = results["navs"]["Min_Variance"].pct_change().dropna()
            
            ttest = stat_tester.paired_ttest(
                lstm_returns, minvar_returns, lstm_key, "Min_Variance"
            )
            print(f"  vs Min_Variance: p-value={ttest['p_value']:.4f}, {ttest['conclusion']}")
            
            sharpe_test = stat_tester.sharpe_ratio_test(
                lstm_returns, minvar_returns, config.risk_free_daily
            )
            print(
                f"  Sharpe diff: {sharpe_test['difference']:.4f} "
                f"({'Significant' if sharpe_test['significant'] else 'Not significant'})"
            )
    
    
    # =======================================================
    # PREPARE FULL PRICE DATA FOR TAX + RISK PROFILES
    # =======================================================
    data_manager_full = DataManager(config)
    price_df_full = data_manager_full.download_data()
    if price_df_full is None or price_df_full.empty:
        print("⚠️ No price data available for tax/profile calculations.")
        full_returns = None
    else:
        full_returns = price_df_full.pct_change().dropna()[config.tickers]
    
    
    # =======================================================
    # SECTION 1: TAX-AWARE PERFORMANCE (LSTM STRATEGY)
    # =======================================================
    print("\n" + "="*70)
    print("TAX-AWARE PERFORMANCE (LSTM STRATEGY)")
    print("="*70 + "\n")
    
    after_tax_results = {}
    
    if full_returns is not None:
        # Map period_name -> test window for slicing
        period_index = {p["name"]: p for p in config.out_of_sample_periods}
        
        for period_name, results in all_results.items():
            if period_name not in period_index:
                continue
            
            # Get test window dates
            pinfo = period_index[period_name]
            test_start = pinfo["test_start"]
            test_end = pinfo["test_end"]
            
            # Slice asset returns for this test window
            period_returns = full_returns.loc[test_start:test_end]
            if period_returns.empty:
                continue
            
            # Get LSTM weight history from results
            weights_hist_all = results.get("weights_history", {})
            if lstm_key not in weights_hist_all:
                continue
            
            hist_list = weights_hist_all[lstm_key]
            if not hist_list:
                continue
            
            # Build weight_history dict: date -> Series of weights
            weight_history = {}
            for entry in hist_list:
                date = entry["date"]
                w_vec = np.array(entry["weights"])
                weight_history[date] = pd.Series(w_vec, index=config.tickers)
            
            # Validate weight_history has dates in the test period
            if not weight_history:
                print(f"  ⚠️ No weight history for {period_name}, skipping tax calculation")
                continue
            
            # Validate overlap with returns period
            weight_dates = set(weight_history.keys())
            return_dates = set(period_returns.index)
            overlap = weight_dates & return_dates
            
            if len(overlap) == 0:
                print(f"  ⚠️ No date overlap between weights and returns for {period_name}")
                continue
            
            # Run tax simulation
            try:
                at_df = simulate_after_tax_returns(
                    returns=period_returns,
                    weight_history=weight_history,
                    tax_rate=getattr(config, "tax_rate", 0.15),
                    harvest=True,
                )
            except Exception as e:
                print(f"  ⚠️ Tax calculation failed for {period_name}: {e}")
                continue
            
            after_tax_results[period_name] = at_df

                        # Get DAILY after-tax NAV by applying daily returns
            # (We need to interpolate or use daily pre-tax NAV as proxy)
            
            # Calculate after-tax metrics using rebalance-period NAV (honest approach)
            # Calculate after-tax metrics using rebalance-period NAV
            # Calculate after-tax metrics using rebalance-period NAV
            # Calculate after-tax metrics using rebalance-period NAV
            at_metrics = PerformanceAnalyzer.compute_metrics_from_nav(
                nav_series=at_df["after_tax"],
                period_length=config.rebalance_freq,
                risk_free_annual=config.risk_free_annual,
            )
            
            # Calculate PRE-TAX metrics - USE DAILY NAV FOR ACCURATE VOLATILITY
            if "navs" in results and lstm_key in results["navs"]:
                daily_nav_pretax = results["navs"][lstm_key]
                
                # Sample pre-tax NAV at rebalance dates only for returns
                rebalance_dates = at_df.index
                sampled_pretax_nav = daily_nav_pretax.loc[daily_nav_pretax.index.isin(rebalance_dates)]
                
                # Pre-tax returns at rebalance frequency
                pretax_reb_returns = sampled_pretax_nav.pct_change().dropna()
                periods_per_year = 252.0 / config.rebalance_freq
                pretax_ann_return = (1.0 + pretax_reb_returns.mean()) ** periods_per_year - 1.0
                
                # After-tax returns at rebalance frequency
                aftertax_reb_returns = at_df["after_tax"].pct_change().dropna()
                aftertax_ann_return = (1.0 + aftertax_reb_returns.mean()) ** periods_per_year - 1.0
                
                # CRITICAL FIX: Use DAILY pre-tax volatility for BOTH Sharpe calculations
                # Taxes reduce returns but don't change the underlying risk
                daily_pretax_returns = daily_nav_pretax.pct_change().dropna()
                daily_pretax_vol = daily_pretax_returns.std() * np.sqrt(252)
                
                # Sharpe ratios using the SAME volatility
                pretax_sharpe = (pretax_ann_return - config.risk_free_annual) / daily_pretax_vol
                aftertax_sharpe = (aftertax_ann_return - config.risk_free_annual) / daily_pretax_vol
    
                # Drawdown from daily NAV (most accurate)
                dd_daily = max_drawdown_from_nav(daily_nav_pretax)
                
                # Calmar ratios
                calmar_pretax = (pretax_ann_return / abs(dd_daily)) if abs(dd_daily) > 1e-9 else np.nan
                calmar_aftertax = (aftertax_ann_return / abs(dd_daily)) if abs(dd_daily) > 1e-9 else np.nan
                
                # Validation: after-tax should be lower than pre-tax
                pretax_final = at_df["gross"].iloc[-1]
                aftertax_final = at_df["after_tax"].iloc[-1]
                if aftertax_final > pretax_final:
                    print(f"⚠️  WARNING: After-tax NAV ({aftertax_final:.4f}) > Pre-tax ({pretax_final:.4f})")
                
                # Print comparison
                print(f"{period_name}:")
                print(f"  Pre-tax Sharpe:      {pretax_sharpe:.3f}")
                print(f"  After-tax Sharpe:    {aftertax_sharpe:.3f}")
                print(f"  Tax drag (Sharpe):   {pretax_sharpe - aftertax_sharpe:.3f}")
                print()
                print(f"  Pre-tax Ann Return:  {pretax_ann_return*100:.2f}%")
                print(f"  After-tax Ann Return: {aftertax_ann_return*100:.2f}%")
                print(f"  Tax drag (Return):   {(pretax_ann_return - aftertax_ann_return)*100:.2f}%")
                print()
                print(f"  Volatility (daily):  {daily_pretax_vol*100:.2f}%")
                print(f"  Max DD (daily):      {dd_daily*100:.2f}%")
                print(f"  Pre-tax Calmar:      {calmar_pretax:.3f}" if np.isfinite(calmar_pretax) else "  Pre-tax Calmar:      NA")
                print(f"  After-tax Calmar:    {calmar_aftertax:.3f}" if np.isfinite(calmar_aftertax) else "  After-tax Calmar:    NA")
            else:
                # Fallback if daily NAV not available
                print(f"{period_name}:")
                print(f"  After-tax Sharpe:    {at_metrics['sharpe']:.3f}")
                print(f"  After-tax Ann Return: {at_metrics['ann_return']*100:.2f}%")
                print(f"  After-tax Ann Vol:   {at_metrics['ann_vol']*100:.2f}%")
                print(f"  Max DD:              {at_metrics['max_dd']*100:.2f}%")
                print(f"  Calmar:              {at_metrics['calmar']:.3f}" if np.isfinite(at_metrics['calmar']) else "  Calmar:              NA")
            
            print("-" * 70)
                         
    # =======================================================
    # SECTION 2: PERSONALISED RISK-PROFILE PORTFOLIOS
    # =======================================================
    print("\n" + "="*70)
    print("PERSONALISED RISK-PROFILE PORTFOLIOS")
    print("="*70 + "\n")
    
    profile_weights_dict = {}
    
    if full_returns is not None:
        # Use last 252 trading days for profile optimisation
        lookback_window = 252
        if full_returns.shape[0] > lookback_window:
            profile_returns = full_returns.iloc[-lookback_window:]
        else:
            profile_returns = full_returns.copy()
        
        # Use the profile chosen at the start (conservative / aggressive)
        active_profile = config.risk_profile
        constraints = get_profile_constraints(active_profile)
        
        profile_weights = optimize_weights_with_profile(
            returns=profile_returns,
            profile_constraints=constraints,
            expected_returns=None,    # or pass LSTM alpha vector here
            method="mean_variance",   # or 'min_variance'
        )
        profile_weights_dict[active_profile] = profile_weights
        
        print(f"{active_profile.capitalize()} profile:")
        for asset, w in profile_weights.items():
            if abs(w) > 1e-4:
                print(f"  {asset:12s}: {w:.3%}")
        print("-" * 50)
        
        # Export to JSON for app front-end
        import json, os
        profile_weights_path = os.path.join(config.live_weights_dir, "risk_profile_weights.json")
        with open(profile_weights_path, "w") as f:
            json.dump({p: w.to_dict() for p, w in profile_weights_dict.items()}, f, indent=2)
        
        print(f"\n✅ Saved risk-profile weights to: {profile_weights_path}")

    
    
    # ----------------- VISUALISATIONS -----------------
    print("\n\nGenerating visualizations...")
    viz = ResearchVisualizer(config)
    viz.plot_out_of_sample_comparison(all_results)
    # NEW: Plot allocation evolution for each period
    lstm_key = f"LSTM_{config.pred_horizons[0]}day"
    for period_name, results in all_results.items():
        if 'weights_history' in results and lstm_key in results['weights_history']:
            viz.plot_allocation_over_time(
                results['weights_history'][lstm_key],
                period_name=f"{period_name}_LSTM"
            )
    
    
    # ----------------- RESULTS SUMMARY (PRE-TAX LSTM) -----------------
    print("\n" + "="*70)
    print("RESULTS SUMMARY (PRE-TAX LSTM)")
    print("="*70 + "\n")
    
    for period_name, results in all_results.items():
        if "metrics" in results and lstm_key in results["metrics"]:
            metrics = results["metrics"][lstm_key]
            print(f"\n{period_name}:")
            print(f"  Sharpe:      {metrics['sharpe']:.3f}")
            print(f"  Ann Return:  {metrics['ann_return']*100:.2f}%")
            print(f"  Ann Vol:     {metrics['ann_vol']*100:.2f}%")
            print(f"  Max DD:      {metrics['max_dd']*100:.2f}%")
            print(f"  Calmar:      {metrics['calmar']:.3f}")
    
    print("\n" + "="*70)
    print(f"✓ COMPLETE! Results in: {config.output_dir}")
    print("="*70)
    print("\n📊 Research Paper: Use metrics & plots from out_of_sample/")
    print("💰 App Development: Use live weights from live_weights/ (incl. risk_profile_weights.json)")
    print("🧾 Tax: Use after-tax NAVs from 'navs_after_tax' in all_results")
    print("\n🚀 READY FOR PRODUCTION!")
    print("="*70 + "\n")


if __name__ == "__main__":
    main()




PRODUCTION-READY PORTFOLIO OPTIMIZATION SYSTEM

App Features:
  ✅ Real transaction costs (commissions + slippage)
  ✅ Intelligent rebalancing (skip if drift < 2%)
  ✅ Asset health validation
  ✅ Crisis mode detection & protection
  ✅ Live portfolio weights export (JSON)
  ✅ Performance attribution analysis
  ✅ Risk alerts & explanations
  ✅ Dollar rounding for real accounts
  ✅ Tax-aware performance (after-tax NAV)
  ✅ Personalized risk-profile portfolios

Ready for Research Paper & Real-Money App!



Using risk profile: Aggressive

PRODUCTION-READY MULTI-ASSET PORTFOLIO SYSTEM
Testing 2 periods
Universe: 40 assets

App Features Enabled:
  • Transaction costs: 7.0 bps
  • Rebalance threshold: 2%
  • Min trade size: 0%
  • Crisis detection: 5% daily vol
  • Asset health checks: Enabled
  • Live weights export: Enabled


DOWNLOADING & VALIDATING MULTI-ASSET DATA
Period: 2015-10-28 to 2025-12-24
Assets: 40

✓ AAPL                 - Healthy
✓ MSFT                 - Healthy
✓ GOOGL        

Bull_Market_2023_2025: 2023-01-03:   0%|                                     | 0/66 [00:00<?, ?it/s]


  Training at 2023-01-03 (40 assets)...


I0000 00:00:1766598748.317073      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1766598752.267733     153 cuda_dnn.cc:529] Loaded cuDNN version 91002


    ✓ AAPL
    ✓ MSFT
    ✓ GOOGL
    ✓ AMZN
    ✓ META
    ✓ NVDA
    ✓ TSLA
    ✓ V
    ✓ JPM
    ✓ WMT
    ✓ RELIANCE
    ✓ TCS
    ✓ HDFCBANK
    ✓ INFY
    ✓ ITC
    ✓ SPY
    ✓ QQQ
    ✓ IWM
    ✓ EFA
    ✓ VTI
    ✓ AGG
    ✓ TLT
    ✓ IEF
    ✓ LQD
    ✓ HYG
    ✓ VNQ
    ✓ O
    ✓ PLD
    ✓ GLD
    ✓ SLV
    ✓ USO
    ✓ DBA
    ✓ XLK
    ✓ XLF
    ✓ XLV
    ✓ XLE
    ✓ VYM
    ✓ SCHD
    ✓ DVY
    ✓ VTV


...

  Skipped 40 rebalances (signal change below threshold)


COVID_Recovery_2020_2022: 2022-10-12:  92%|████████████▊ | 67/73 [07:33<00:16,  2.79s/it, NAV=1.669]

  Skipped 45 rebalances (signal change below threshold)


COVID_Recovery_2020_2022: 2022-12-27: 100%|██████████████| 73/73 [07:49<00:00,  6.44s/it, NAV=1.626]



⚠️ Crisis mode triggered 2 times
💰 Saved 49 rebalances (below 2% threshold)

OUT-OF-SAMPLE PERFORMANCE COMPARISON

LSTM Performance Across Periods:
----------------------------------------------------------------------
                            sharpe  ann_return   ann_vol    max_dd    calmar
Bull_Market_2023_2025     1.691831    0.515883  0.216535 -0.244640  2.108742
COVID_Recovery_2020_2022  0.623669    0.222920  0.242631 -0.316876  0.703491


Average Out-of-Sample Performance:
----------------------------------------------------------------------
sharpe        1.157750
ann_return    0.369401
ann_vol       0.229583
max_dd       -0.280758
dtype: float64

STATISTICAL TESTING


Bull_Market_2023_2025:
----------------------------------------------------------------------
  vs Min_Variance: p-value=0.0079, Significantly different
  Sharpe diff: 0.9473 (Not significant)

COVID_Recovery_2020_2022:
----------------------------------------------------------------------
  vs Min_Variance: p